# SemEval-2026 Task 13A — AI Code Detection Pipeline

## Kaggle Quick Start
```
1. File > Add Shortcut to Drive (nếu Colab) hoặc Add Data (nếu Kaggle)
2. Chạy Cell 1 để clone repo và cài deps
3. Chỉnh KAGGLE_INPUT trong Cell 2 nếu cần
4. Chạy các cell theo thứ tự
```

In [ ]:
# ── CELL 1: Clone repo & cài dependencies ───────────────────────────────
import os, sys

REPO_URL = "https://github.com/YOUR_USERNAME/semeval.git"  # <- đổi URL
REPO_DIR = "/kaggle/working/semeval" if os.path.exists("/kaggle") else "/content/semeval"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    print(f"Cloned to {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")
    print("Repo already exists, pulled latest")

# Thêm vào Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Cài dependencies
os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")
print("Dependencies installed")

In [ ]:
# ── CELL 2: Config ───────────────────────────────────────────────────────
import os, sys, logging
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s')

from config import CFG
cfg = CFG()

# Kaggle: đổi slug dataset của bạn ở đây nếu cần
# cfg.train_data = "/kaggle/input/YOUR-DATASET/train.parquet"
# cfg.val_data   = "/kaggle/input/YOUR-DATASET/validation.parquet"
# cfg.test_data  = "/kaggle/input/YOUR-DATASET/test.parquet"

os.makedirs(cfg.output_dir, exist_ok=True)
print(f"Environment : {cfg.env}")
print(f"GPU available: {cfg.fp16}")
print(f"Train data  : {cfg.train_data}")
print(f"Output dir  : {cfg.output_dir}")
print(f"Batch size  : {cfg.batch_size} (effective: {cfg.batch_size * cfg.grad_accum})")

In [ ]:
# ── CELL 3: Feature Extraction ───────────────────────────────────────────
from feature_extractor import extract_all_features
from data_utils import load_dataframe

TRAIN_FEAT = cfg.output_dir + "/train_features.parquet"
VAL_FEAT   = cfg.output_dir + "/val_features.parquet"
TEST_FEAT  = cfg.output_dir + "/test_features.parquet" if cfg.test_data else None

if not os.path.exists(TRAIN_FEAT):
    print("Extracting train features...")
    train_df = load_dataframe(cfg.train_data, cfg.max_train, "Train", cfg.seed)
    train_feat = extract_all_features(train_df['code'], show_progress=True)
    train_feat['label'] = train_df['label'].values
    train_feat.to_parquet(TRAIN_FEAT, index=False)
    print(f"Train features: {train_feat.shape} -> {TRAIN_FEAT}")
else:
    print(f"Train features already exist: {TRAIN_FEAT}")

if not os.path.exists(VAL_FEAT):
    print("Extracting val features...")
    val_df = load_dataframe(cfg.val_data, cfg.max_val, "Val", cfg.seed)
    val_feat = extract_all_features(val_df['code'], show_progress=True)
    val_feat['label'] = val_df['label'].values
    val_feat.to_parquet(VAL_FEAT, index=False)
    print(f"Val features: {val_feat.shape} -> {VAL_FEAT}")
else:
    print(f"Val features already exist: {VAL_FEAT}")

In [ ]:
# ── CELL 4: OOD Shift Diagnosis ──────────────────────────────────────────
# (Chỉ chạy nếu có test features)
if TEST_FEAT and os.path.exists(TEST_FEAT):
    from diag_shift import compute_shift_report, plot_cohens_d
    shift_df = compute_shift_report(
        TRAIN_FEAT, TEST_FEAT, top_n=20,
        save_csv=cfg.output_dir + "/shift_report.csv"
    )
    plot_cohens_d(shift_df, top_n=25,
                  save_path=cfg.output_dir + "/cohens_d.png")
    display(shift_df.head(15))
else:
    print("No test features for OOD diagnosis. Skipping.")

In [ ]:
# ── CELL 5: Train GBM Ensemble ───────────────────────────────────────────
from train_gbm import run_gbm

gbm_tau, val_proba_gbm = run_gbm(
    train_feat_path = TRAIN_FEAT,
    val_feat_path   = VAL_FEAT,
    cfg             = cfg,
    test_feat_path  = TEST_FEAT,
)
print(f"GBM done. tau={gbm_tau:.3f}")

In [ ]:
# ── CELL 6: Fine-tune CodeBERT ────────────────────────────────────────────
from train_codebert import run_codebert

codebert_tau, val_proba_codebert = run_codebert(cfg)
print(f"CodeBERT done. tau={codebert_tau:.3f}")

In [ ]:
# ── CELL 7: Train IF+CNB ──────────────────────────────────────────────────
from train_ifcnb import run_ifcnb
from data_utils import load_dataframe

train_df_full = load_dataframe(cfg.train_data, cfg.max_train, "Train", cfg.seed)
val_df_full   = load_dataframe(cfg.val_data, cfg.max_val, "Val", cfg.seed)
test_df_full  = pd.read_parquet(cfg.test_data) if cfg.test_data and os.path.exists(cfg.test_data) else None

val_proba_ifcnb = run_ifcnb(train_df_full, val_df_full, cfg, test_df=test_df_full)
print(f"IF+CNB done. mean proba={val_proba_ifcnb.mean():.3f}")

In [ ]:
# ── CELL 8: Soft-Voting Ensemble ─────────────────────────────────────────
from ensemble import run_ensemble
from data_utils import load_dataframe

val_full = load_dataframe(cfg.val_data, None, "Full-Val", cfg.seed)
y_val = val_full['label'].values

def try_load(path):
    return np.load(path) if path and os.path.exists(path) else None

val_blend = run_ensemble(
    val_proba_gbm       = val_proba_gbm,
    val_proba_codebert  = val_proba_codebert,
    val_proba_ifcnb     = val_proba_ifcnb,
    y_val               = y_val,
    test_proba_gbm      = try_load(cfg.output_dir + "/test_proba_gbm.npy"),
    test_proba_codebert = try_load(cfg.output_dir + "/test_proba_codebert.npy"),
    test_proba_ifcnb    = try_load(cfg.output_dir + "/test_proba_ifcnb.npy"),
    test_ids            = pd.read_parquet(cfg.test_data)['ID'].tolist() if cfg.test_data and os.path.exists(cfg.test_data) else None,
    output_dir          = cfg.output_dir,
    submission_out      = cfg.submission_out,
)
print(f"Submission saved: {cfg.submission_out}")

In [ ]:
# ── CELL 9: Summary ───────────────────────────────────────────────────────
from metrics import optimize_threshold
from sklearn.metrics import f1_score

results = {
    'GBM':      val_proba_gbm,
    'CodeBERT': val_proba_codebert,
    'IF+CNB':   val_proba_ifcnb,
    'Ensemble': val_blend,
}
print('\n' + '='*50)
print(f'{"Model":<15} {"Best tau":>9} {"Val Macro F1":>13}')
print('-'*50)
for name, proba in results.items():
    tau, f1 = optimize_threshold(proba, y_val)
    print(f'{name:<15} {tau:>9.3f} {f1:>13.4f}')
print('='*50)